# 21｜Transformer加速_推論與參數高效微調

量化 KV cache 的計算差異，並計算 LoRA 的可訓練參數量。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
lengths = np.array([16, 32, 64, 128, 256])
without_cache = lengths * (lengths + 1) / 2
with_cache = lengths
d_in = d_out = 4096; rank = 8
full = d_in * d_out
lora = rank * (d_in + d_out)
table = pd.DataFrame({'sequence_length': lengths,
                      'projected_token_states_without_cache': without_cache.astype(int),
                      'with_KV_cache': with_cache,
                      'ratio': without_cache / with_cache})
print(f'LoRA trainable parameters: {lora:,} vs full matrix {full:,} ({lora/full:.3%})')
table

## 執行結果圖

In [ ]:
plt.loglog(lengths, without_cache, 'o-', label='recompute prefix')
plt.loglog(lengths, with_cache, 's-', label='KV cache')
plt.xlabel('generated sequence length'); plt.ylabel('projected token states')
plt.title('KV cache avoids repeated key/value projections'); plt.legend()
savefig('21_transformer_efficiency_demo')
report('transformer_efficiency', {'lora_params': lora, 'full_params': full, 'table': table.to_dict('records')})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。